# 03 - Database Loading

## Objective

The goal of this notebook is to load the cleaned real estate dataset into a PostgreSQL database using a normalized schema.

The loading process follows the order below:

1. Status
2. Brokers
3. Locations
4. Properties

Finally, the data load will be validated to ensure data integrity.

### 1. Import libraries

In [3]:
import numpy as np
import pandas as pd
import os
from sqlalchemy import create_engine

In [4]:
!pip install python_dotenv

In [5]:
from dotenv import load_dotenv

In [6]:
!pip install psycopg2-binary

In [7]:
import psycopg2

### 2. Load Clean Dataset

In [8]:
df = pd.read_csv("../data/processed/usa_real_estate_clean.csv",  dtype={"zip_code": str})

In [9]:
df.head()

,brokered_by,status,price,bedrooms,bathrooms,acre_lot,street,city,state,zip_code,house_size,prev_sold_date,lot_size_sqft,price_per_sqft
0,103378.0,for_sale,105000.0,3.0,2.0,0.12,1962661.0,Adjuntas,Puerto Rico,00601,920.0,NaN,5227.2,114.130435
1,52707.0,for_sale,80000.0,4.0,2.0,0.08,1902874.0,Adjuntas,Puerto Rico,00601,1527.0,NaN,3484.8,52.390308
2,103379.0,for_sale,67000.0,2.0,1.0,0.15,1404990.0,Juana Diaz,Puerto Rico,00795,748.0,NaN,6534.0,89.572193
3,31239.0,for_sale,145000.0,4.0,2.0,0.10,1947675.0,Ponce,Puerto Rico,00731,1800.0,NaN,4356.0,80.555556
4,34632.0,for_sale,65000.0,6.0,2.0,0.05,331151.0,Mayaguez,Puerto Rico,00680,NaN,NaN,2178.0,NaN


In [10]:
df.shape

(2224460, 14)

### 3. Connect to PostgreSQL

In [11]:
load_dotenv()

True

In [12]:
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

In [13]:
engine = create_engine( f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

### 4. Load Status Dimension

##### Create Status DataFrame

In [14]:
status_df = (df[['status']].drop_duplicates().sort_values('status').reset_index(drop=True))
status_df

,status
0,for_sale
1,ready_to_build
2,sold


##### Load Status into PostgreSQL

In [16]:
status_df.to_sql("status", con=engine, if_exists="append", index=False)

3

In [17]:
pd.read_sql("Select *  from status;", engine)

,status_id,status
0,1,for_sale
1,2,ready_to_build
2,3,sold


### 5. Load Brokers Dimension

##### Create Brokers DataFrame

In [18]:
brokers_df = df[['brokered_by']].dropna().drop_duplicates().sort_values('brokered_by').reset_index(drop=True)

In [19]:
brokers_df

,brokered_by
0,0.0
1,1.0
2,2.0
3,3.0
4,4.0
...,...
110112,110138.0
110113,110139.0
110114,110140.0
110115,110141.0


##### Load Brokers into PostgreSQL

In [20]:
brokers_df.rename(columns = {'brokered_by':'broker_id'}, inplace=True)

In [21]:
brokers_df

,broker_id
0,0.0
1,1.0
2,2.0
3,3.0
4,4.0
...,...
110112,110138.0
110113,110139.0
110114,110140.0
110115,110141.0


In [22]:
brokers_df.to_sql("brokers", con=engine, if_exists="append", index=False)

117

In [23]:
pd.read_sql("select * from brokers;", engine)

,broker_id
0,0
1,1
2,2
3,3
4,4
...,...
110112,110138
110113,110139
110114,110140
110115,110141


### 6. Load Locations Dimension

##### Create Locations DataFrame

In [24]:
locations_df = df[['street', 'city', 'state', 'zip_code']].drop_duplicates()

In [25]:
locations_df.head()

,street,city,state,zip_code
0,1962661.0,Adjuntas,Puerto Rico,00601
1,1902874.0,Adjuntas,Puerto Rico,00601
2,1404990.0,Juana Diaz,Puerto Rico,00795
3,1947675.0,Ponce,Puerto Rico,00731
4,331151.0,Mayaguez,Puerto Rico,00680


In [26]:
locations_df.isnull().sum()

street      4249
city        1314
state          4
zip_code     252
dtype: int64

In [27]:
locations_df = df[['street', 'city', 'state', 'zip_code']].drop_duplicates().sort_values(['state', 'city', 'street']).reset_index(drop=True)

In [28]:
locations_df.head()

,street,city,state,zip_code
0,31914.0,Abbeville,Alabama,36310
1,65459.0,Abbeville,Alabama,36310
2,99403.0,Abbeville,Alabama,36310
3,259362.0,Abbeville,Alabama,36310
4,341424.0,Abbeville,Alabama,36310


In [29]:
locations_df.shape

(2049594, 4)

In [30]:
locations_df.isnull().sum()

street      4249
city        1314
state          4
zip_code     252
dtype: int64

In [31]:
len(df)

2224460

In [32]:
len(locations_df)

2049594

In [33]:
locations_df.duplicated(subset=['street', 'city', 'state', 'zip_code']).sum()

np.int64(0)

In [34]:
locations_df.isna().sum()

street      4249
city        1314
state          4
zip_code     252
dtype: int64

In [35]:
locations_df.to_sql("locations", engine, if_exists="append", index=False)

594

In [36]:
pd.read_sql("select * from locations limit 10;", engine)

,location_id,street,city,state,zip_code
0,1,31914,Abbeville,Alabama,36310
1,2,65459,Abbeville,Alabama,36310
2,3,99403,Abbeville,Alabama,36310
3,4,259362,Abbeville,Alabama,36310
4,5,341424,Abbeville,Alabama,36310
5,6,374797,Abbeville,Alabama,36310
6,7,410796,Abbeville,Alabama,36310
7,8,475573,Abbeville,Alabama,36310
8,9,602278,Abbeville,Alabama,36310
9,10,605137,Abbeville,Alabama,36310


### 7. Load Properties Dimension

In [37]:
# Reading locations table from PostgreSQL
locations_lookup = pd.read_sql("""SELECT * FROM locations;""", engine)
locations_lookup.head()

,location_id,street,city,state,zip_code
0,1,31914.0,Abbeville,Alabama,36310
1,2,65459.0,Abbeville,Alabama,36310
2,3,99403.0,Abbeville,Alabama,36310
3,4,259362.0,Abbeville,Alabama,36310
4,5,341424.0,Abbeville,Alabama,36310


In [38]:
print(df['street'].dtype)
print(locations_lookup['street'].dtype)

print(df['zip_code'].dtype)
print(locations_lookup['zip_code'].dtype)

float64
float64
object
object


In [39]:
# Merging locations_lookup with properties
properties_df = df.merge(locations_lookup, on=["street", "city", "state", "zip_code"], how="left")

In [40]:
properties_df.head()

,brokered_by,status,price,bedrooms,bathrooms,acre_lot,street,city,state,zip_code,house_size,prev_sold_date,lot_size_sqft,price_per_sqft,location_id
0,103378.0,for_sale,105000.0,3.0,2.0,0.12,1962661.0,Adjuntas,Puerto Rico,00601,920.0,NaN,5227.2,114.130435,1577620
1,52707.0,for_sale,80000.0,4.0,2.0,0.08,1902874.0,Adjuntas,Puerto Rico,00601,1527.0,NaN,3484.8,52.390308,1577619
2,103379.0,for_sale,67000.0,2.0,1.0,0.15,1404990.0,Juana Diaz,Puerto Rico,00795,748.0,NaN,6534.0,89.572193,1579107
3,31239.0,for_sale,145000.0,4.0,2.0,0.10,1947675.0,Ponce,Puerto Rico,00731,1800.0,NaN,4356.0,80.555556,1579505
4,34632.0,for_sale,65000.0,6.0,2.0,0.05,331151.0,Mayaguez,Puerto Rico,00680,NaN,NaN,2178.0,NaN,1579288


In [41]:
properties_df["location_id"].isna().sum()

np.int64(0)

In [42]:
# Reading brokers table from PostgreSQL
brokers_lookup = pd.read_sql("""SELECT * FROM brokers;""", engine)
brokers_lookup.head()

,broker_id
0,0
1,1
2,2
3,3
4,4


In [43]:
# Merging brokers_lookup with properties
properties_df=properties_df.merge(brokers_lookup, left_on='brokered_by', right_on='broker_id', how='left')
properties_df.head()

,brokered_by,status,price,bedrooms,bathrooms,acre_lot,street,city,state,zip_code,house_size,prev_sold_date,lot_size_sqft,price_per_sqft,location_id,broker_id
0,103378.0,for_sale,105000.0,3.0,2.0,0.12,1962661.0,Adjuntas,Puerto Rico,00601,920.0,NaN,5227.2,114.130435,1577620,103378.0
1,52707.0,for_sale,80000.0,4.0,2.0,0.08,1902874.0,Adjuntas,Puerto Rico,00601,1527.0,NaN,3484.8,52.390308,1577619,52707.0
2,103379.0,for_sale,67000.0,2.0,1.0,0.15,1404990.0,Juana Diaz,Puerto Rico,00795,748.0,NaN,6534.0,89.572193,1579107,103379.0
3,31239.0,for_sale,145000.0,4.0,2.0,0.10,1947675.0,Ponce,Puerto Rico,00731,1800.0,NaN,4356.0,80.555556,1579505,31239.0
4,34632.0,for_sale,65000.0,6.0,2.0,0.05,331151.0,Mayaguez,Puerto Rico,00680,NaN,NaN,2178.0,NaN,1579288,34632.0


In [44]:
properties_df["broker_id"].isna().sum()

np.int64(4529)

In [45]:
# Reading status from PostgreSQL
status_lookup = pd.read_sql("""SELECT * FROM status;""", engine)
status_lookup.head()

,status_id,status
0,1,for_sale
1,2,ready_to_build
2,3,sold


In [46]:
# Merging status_lookup with properties_df
properties_df = properties_df.merge(status_lookup, on=['status'], how='left')
properties_df.head()

,brokered_by,status,price,bedrooms,bathrooms,acre_lot,street,city,state,zip_code,house_size,prev_sold_date,lot_size_sqft,price_per_sqft,location_id,broker_id,status_id
0,103378.0,for_sale,105000.0,3.0,2.0,0.12,1962661.0,Adjuntas,Puerto Rico,00601,920.0,NaN,5227.2,114.130435,1577620,103378.0,1
1,52707.0,for_sale,80000.0,4.0,2.0,0.08,1902874.0,Adjuntas,Puerto Rico,00601,1527.0,NaN,3484.8,52.390308,1577619,52707.0,1
2,103379.0,for_sale,67000.0,2.0,1.0,0.15,1404990.0,Juana Diaz,Puerto Rico,00795,748.0,NaN,6534.0,89.572193,1579107,103379.0,1
3,31239.0,for_sale,145000.0,4.0,2.0,0.10,1947675.0,Ponce,Puerto Rico,00731,1800.0,NaN,4356.0,80.555556,1579505,31239.0,1
4,34632.0,for_sale,65000.0,6.0,2.0,0.05,331151.0,Mayaguez,Puerto Rico,00680,NaN,NaN,2178.0,NaN,1579288,34632.0,1


In [47]:
properties_df["status_id"].isna().sum()

np.int64(0)

In [48]:
properties_df[['broker_id', 'status_id', 'location_id']].isna().sum()

broker_id      4529
status_id         0
location_id       0
dtype: int64

In [50]:
# Creating properties_final
properties_final = properties_df[['broker_id', 
                                  'status_id', 
                                  'location_id', 
                                  'price', 
                                  'bedrooms', 
                                  'bathrooms', 
                                  'acre_lot', 
                                  'house_size', 
                                  'prev_sold_date', 
                                  'price_per_sqft']]

In [51]:
properties_final.head()

,broker_id,status_id,location_id,price,bedrooms,bathrooms,acre_lot,house_size,prev_sold_date,price_per_sqft
0,103378.0,1,1577620,105000.0,3.0,2.0,0.12,920.0,NaN,114.130435
1,52707.0,1,1577619,80000.0,4.0,2.0,0.08,1527.0,NaN,52.390308
2,103379.0,1,1579107,67000.0,2.0,1.0,0.15,748.0,NaN,89.572193
3,31239.0,1,1579505,145000.0,4.0,2.0,0.10,1800.0,NaN,80.555556
4,34632.0,1,1579288,65000.0,6.0,2.0,0.05,NaN,NaN,NaN


In [52]:
properties_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2224460 entries, 0 to 2224459
Data columns (total 10 columns):
 #   Column          Dtype  
---  ------          -----  
 0   broker_id       float64
 1   status_id       int64  
 2   location_id     int64  
 3   price           float64
 4   bedrooms        float64
 5   bathrooms       float64
 6   acre_lot        float64
 7   house_size      float64
 8   prev_sold_date  object 
 9   price_per_sqft  float64
dtypes: float64(7), int64(2), object(1)
memory usage: 169.7+ MB


In [53]:
properties_final.isnull().sum()

broker_id           4529
status_id              0
location_id            0
price                  0
bedrooms          480816
bathrooms         510688
acre_lot          324860
house_size        567868
prev_sold_date    732925
price_per_sqft    567868
dtype: int64

In [54]:
properties_final.shape

(2224460, 10)

##### Load Properties into PostgreSQL

In [55]:
properties_final.to_sql("properties", engine, if_exists="append", index=False)

460

In [56]:
pd.read_sql("""SELECT * FROM properties LIMIT 10;""", engine)

,property_id,broker_id,location_id,status_id,price,bedrooms,bathrooms,acre_lot,house_size,prev_sold_date,price_per_sqft
0,1,103378,1577620,1,105000.0,3,2,0.12,920.0,None,114.13
1,2,52707,1577619,1,80000.0,4,2,0.08,1527.0,None,52.39
2,3,103379,1579107,1,67000.0,2,1,0.15,748.0,None,89.57
3,4,31239,1579505,1,145000.0,4,2,0.10,1800.0,None,80.56
4,5,34632,1579288,1,65000.0,6,2,0.05,NaN,None,NaN
5,6,103378,1580241,1,179000.0,4,3,0.46,2520.0,None,71.03
6,7,1205,1578444,1,50000.0,3,1,0.20,2040.0,None,24.51
7,8,50739,1579468,1,71600.0,3,2,0.08,1050.0,None,68.19
8,9,81909,1579455,1,100000.0,2,1,0.09,1092.0,None,91.58
9,10,65672,1579171,1,300000.0,5,3,7.46,5403.0,None,55.52


In [57]:
pd.read_sql("""SELECT MAX(property_id) FROM properties;""", engine)

,max
0,2224460


### 8. Data Validation

In [59]:
# Validate row count
pd.read_sql("""
    SELECT COUNT(*) AS total_properties
    FROM properties;
""", engine)

,total_properties
0,2224460


In [60]:
# Validate required foreign keys
pd.read_sql("""
    SELECT
        COUNT(*) FILTER (WHERE location_id IS NULL) AS null_location_id,
        COUNT(*) FILTER (WHERE status_id IS NULL) AS null_status_id
    FROM properties;
""", engine)

,null_location_id,null_status_id
0,0,0


In [61]:
pd.read_sql("""
    SELECT COUNT(*) AS null_broker_id
    FROM properties
    WHERE broker_id IS NULL;
""", engine)

,null_broker_id
0,4529


In [62]:
pd.read_sql("""
SELECT
    COUNT(*) FILTER (
        WHERE p.broker_id IS NOT NULL AND b.broker_id IS NULL
    ) AS invalid_broker_fk,

    COUNT(*) FILTER (
        WHERE l.location_id IS NULL
    ) AS invalid_location_fk,

    COUNT(*) FILTER (
        WHERE s.status_id IS NULL
    ) AS invalid_status_fk

FROM properties p

LEFT JOIN brokers b
    ON p.broker_id = b.broker_id

LEFT JOIN locations l
    ON p.location_id = l.location_id

LEFT JOIN status s
    ON p.status_id = s.status_id;
""", engine)

,invalid_broker_fk,invalid_location_fk,invalid_status_fk
0,0,0,0


## ETL Validation and Summary

The cleaned real estate dataset was successfully transformed and loaded into the PostgreSQL relational database.

### Validation Results

- 2,224,460 property records were successfully loaded.
- The source and destination row counts match.
- All properties were successfully mapped to a valid `location_id`.
- All properties were successfully mapped to a valid `status_id`.
- 4,529 properties have no associated broker, consistent with missing broker information in the source dataset.
- No invalid foreign key references were found for brokers, locations, or status.

The ETL process successfully preserved the source data while maintaining the integrity of the relational model.